### Phần 1: Tổng quan tài liệu (Literature Review)

Bộ dữ liệu **Pima Indians Diabetes** của NIDDK là một trong những cơ sở dữ liệu y sinh kinh điển nhất về dự đoán tiểu đường. Cột mốc nổi bật nhất là nghiên cứu của *Smith et al.* (1988), nơi họ đã phát triển thuật toán ADAP để chẩn đoán đái tháo đường khởi phát. Tuy nhiên, đóng góp cốt lõi của họ không chỉ nằm ở thuật toán mà còn ở tư duy **Tiền xử lý (Data Preprocessing)** vượt thời đại.

Trong bộ dữ liệu thô, vô số chỉ số sinh tồn của bệnh nhân - như Huyết áp tâm trương, Chỉ số khối cơ thể (BMI) và Nồng độ Glucose - bị ghi nhận bằng `0`. Ở góc độ lâm sàng, không một cơ thể sống nào có BMI hay Huyết áp bằng 0. Nếu thuật toán tiếp nhận các giá trị này như một đo lường thông thường, nó sẽ bóp méo hoàn toàn bộ trọng số (weights) và triệt tiêu tính chính xác y khoa. Vì vậy, các tác giả đã nhạy bén xác định đây là những dữ liệu khuyết thiếu ngầm định (Missing Values). Quá trình bắt buộc loại bỏ và dùng phương pháp nội suy để bù đắp các số 0 này chính là bước sống còn nhằm bảo vệ tính toàn vẹn sinh lý học của bài toán dự đoán.

**Định nghĩa biến đo lường:**
- **Input (Đầu vào):** 8 biến lâm sàng (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age).
- **Output (Biến mục tiêu):** `Outcome` (0: Âm tính/Khỏe mạnh, 1: Dương tính/Tiểu đường).

### Phần 2: Load Data Động & Khám phá sơ bộ

In [ ]:
import os
import re
import glob
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 2.1 Trích xuất động tên cột từ file metadata (.names/.txt)
# ==========================================
dataset_dir = 'dataset/'
search_pattern = os.path.join(dataset_dir, '*pima*names*')
# Glob tìm tự động bất kỳ file nào khớp tên
matched_files = glob.glob(search_pattern)

if not matched_files:
    raise FileNotFoundError(f"Không tìm thấy file metadata khớp với pattern '{search_pattern}'")

names_file_path = matched_files[0]
print(f"Đang trích xuất metadata từ: {names_file_path}")

extracted_names = []
with open(names_file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

capture = False
for line in lines:
    if line.startswith('7. For Each Attribute:'):
        capture = True
        continue
    if capture and line.startswith('8. Missing Attribute Values:'):
        break
    if capture:
        match = re.search(r'\s*\d+\.\s+(.*)', line)
        if match:
            raw_name = match.group(1).strip().lower()
            # Bóc tách Regex và map thành chuẩn tên biến lâm sàng siêu ngắn gọn
            if 'pregnant' in raw_name: col = 'Pregnancies'
            elif 'glucose' in raw_name: col = 'Glucose'
            elif 'blood pressure' in raw_name: col = 'BloodPressure'
            elif 'skin fold' in raw_name: col = 'SkinThickness'
            elif 'insulin' in raw_name: col = 'Insulin'
            elif 'body mass index' in raw_name: col = 'BMI'
            elif 'pedigree' in raw_name: col = 'DiabetesPedigreeFunction'
            elif 'age' in raw_name: col = 'Age'
            elif 'class' in raw_name: col = 'Outcome'
            else: col = match.group(1).strip()
            extracted_names.append(col)

print("Map thành công 9 biến:", extracted_names)

# ==========================================
# 2.2 Load CSV & Thống kê Missing ảo
# ==========================================
df = pd.read_csv('dataset/pima-indians-diabetes.csv', header=None, names=extracted_names)

print("\nHiển thị 5 dòng gốc chưa qua xử lý:")
display(df.head())

features_to_check = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("\n!!! CẢNH BÁO LỖI Y KHOA: Lượng giá trị 0 (Missing ảo) ở 5 cột nhạy cảm:")
for col in features_to_check:
    count_zeros = (df[col] == 0).sum()
    print(f"- {col}: {count_zeros} dòng bất thường")

### Phần 3: Tiền xử lý dữ liệu (Data Cleaning & Scaling)

In [ ]:
# 1. Gán số 0 phi vật lý thành NaN
df_cleaned = df.copy()
df_cleaned[features_to_check] = df_cleaned[features_to_check].replace(0, np.nan)

# 2. Lấp dữ liệu bằng SimpleImputer (Chiến lược Median để nội suy trung vị)
imputer = SimpleImputer(strategy='median')
df_cleaned[features_to_check] = imputer.fit_transform(df_cleaned[features_to_check])

# 3. LƯU BẢN SAO LÂM SÀNG TỰ NHIÊN: Dùng cho các Case Study phân loại tuổi, BMI, Glucose theo mức y tế gốc
df_clinical = df_cleaned.copy()
input_cols = extracted_names[:-1]
outcome_col = extracted_names[-1]

# 4. Chuẩn hóa bằng StandardScaler: Dùng cho các biểu đồ tương quan và đo lường khoảng cách
scaler = StandardScaler()
df_scaled = df_cleaned.copy()
df_scaled[input_cols] = scaler.fit_transform(df_cleaned[input_cols])

print("5 dòng đầu của dữ liệu Đã Nội Suy (Cleaned & Scaled):")
display(df_scaled.head())

### Phần 4: KHÁM PHÁ DỮ LIỆU CHUYÊN SÂU (Dựa trên 7 Case Study lâm sàng)

*(Hoàn toàn KHÔNG dùng Machine Learning dự đoán. Mọi đồ thị sẽ được xuất ra `.png`)*

In [ ]:
# 4.0 Thiết lập cơ chế xuất ảnh động
output_dir = 'images/'
os.makedirs(output_dir, exist_ok=True)
img_counter = 1

def save_plot(filename_suffix):
    """Lưu biểu đồ chất lượng cao với tên tuần tự"""
    global img_counter
    filepath = os.path.join(output_dir, f"{img_counter:02d}_{filename_suffix}.png")
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()  # Khai tử biểu đồ sau khi xuất ảnh để chống giật LAG / Tràn RAM
    img_counter += 1

sns.set_theme(style="whitegrid", palette="muted")

###################################################################
# CASE 1: PHÂN TÍCH BIẾN MỤC TIÊU (OUTCOME)
###################################################################
plt.figure(figsize=(6,6))
outcome_counts = df_clinical['Outcome'].value_counts()
plt.pie(outcome_counts, labels=['Khỏe mạnh (0)', 'Mắc bệnh (1)'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=140, explode=[0, 0.1])
plt.title("Tỷ lệ Mắc bệnh vs Khỏe mạnh trong quần thể Pima")
save_plot("Case1_Outcome_PieChart")

plt.figure(figsize=(6,5))
sns.countplot(x='Outcome', data=df_clinical, palette=['#2ecc71', '#e74c3c'])
plt.title("Biểu đồ cột: Số lượng Mắc bệnh (1) vs Khỏe mạnh (0)")
save_plot("Case1_Outcome_CountPlot")

# Radar Radar hoặc Barplot so sánh 8 biến. Do các biến chênh lệch đơn vị, ta buộc vẽ trên df_scaled!
plt.figure(figsize=(10,6))
mean_values = df_scaled.groupby('Outcome')[input_cols].mean().reset_index()
mean_values_melt = pd.melt(mean_values, id_vars='Outcome', var_name='Biến Input', value_name='TB Chuẩn hóa (Z-Score)')
sns.barplot(data=mean_values_melt, x='Biến Input', y='TB Chuẩn hóa (Z-Score)', hue='Outcome', palette=['#2ecc71', '#e74c3c'])
plt.xticks(rotation=45)
plt.title("Mô hình Barplot Tương quan Z-Score của 8 biến sinh lý theo Outcome")
save_plot("Case1_Barplot_Means_ZScore")

###################################################################
# CASE 2: PHÂN TÍCH THEO NHÓM TUỔI (AGE)
###################################################################
bins_age = [0, 30, 40, 50, 200]
labels_age = ['<30', '31-40', '41-50', '>50']
df_clinical['Age_Group'] = pd.cut(df_clinical['Age'], bins=bins_age, labels=labels_age, right=False)

plt.figure(figsize=(8,5))
sns.barplot(x='Age_Group', y='Outcome', data=df_clinical, ci=None, color='#3498db')
plt.title("Tốc độ gia tăng rủi ro mắc đái tháo đường theo Tầng Tuổi")
plt.ylabel("Tỷ lệ dương tính (Mean Outcome)")
save_plot("Case2_Age_DiseaseRate_Barplot")

plt.figure(figsize=(8,5))
sns.lineplot(x='Age', y='BMI', data=df_clinical, hue='Outcome', ci=None)
plt.title("Mô hình động học BMI biến thiên theo Tuổi")
save_plot("Case2_Age_BMI_Lineplot")

plt.figure(figsize=(8,5))
sns.lineplot(x='Age', y='Glucose', data=df_clinical, hue='Outcome', ci=None)
plt.title("Độ bốc Đồng đẳng Glucose trên Thước đo Tuổi tác")
save_plot("Case2_Age_Glucose_Lineplot")

plt.figure(figsize=(8,5))
sns.lineplot(x='Age', y='BloodPressure', data=df_clinical, hue='Outcome', ci=None)
plt.title("Trọng lượng Huyết áp chịu tác động của Tuổi thọ")
save_plot("Case2_Age_BloodPressure_Lineplot")

###################################################################
# CASE 3: PHÂN TÍCH BMI (CHUẨN WHO)
###################################################################
bins_bmi = [0, 18.5, 25, 30, 200]
labels_bmi = ['Gầy (<18.5)', 'Bình thường (18.5-24.9)', 'Thừa cân (25-29.9)', 'Béo phì (>=30)']
df_clinical['BMI_Group'] = pd.cut(df_clinical['BMI'], bins=bins_bmi, labels=labels_bmi, right=False)

plt.figure(figsize=(8,5))
sns.barplot(x='BMI_Group', y='Outcome', data=df_clinical, ci=None, palette='magma')
plt.title("Thiết lập tỷ lệ đái tháo đường qua Tầm soát BMI của WHO")
plt.ylabel("Tỷ lệ mắc bệnh (Mean Outcome)")
save_plot("Case3_BMI_WHO_Rate")

plt.figure(figsize=(8,5))
sns.scatterplot(x='BMI', y='Glucose', hue='Outcome', data=df_clinical, palette=['#3498db', '#e74c3c'], alpha=0.7)
plt.title("Tương quan cộng hưởng giữa Cân nặng (BMI) và Nồng độ Đường (Glucose)")
save_plot("Case3_Scatter_BMI_Glucose")

###################################################################
# CASE 4: PHÂN TÍCH GLUCOSE & INSULIN (CHUẨN ADA)
###################################################################
bins_glu = [0, 100, 126, 1000]
labels_glu = ['Bình thường (<100)', 'Tiền tiểu đường (100-125)', 'Tiểu đường (>=126)']
df_clinical['Glucose_Group'] = pd.cut(df_clinical['Glucose'], bins=bins_glu, labels=labels_glu, right=False)

plt.figure(figsize=(8,5))
sns.barplot(x='Glucose_Group', y='Outcome', data=df_clinical, ci=None, palette='viridis')
plt.title("Phân bố Glucose theo chuẩn ADA và Xác suất chuyển thành bệnh lý")
save_plot("Case4_Glucose_ADA_Rate")

plt.figure(figsize=(8,5))
sns.scatterplot(x='Insulin', y='Glucose', hue='Outcome', data=df_clinical, palette=['#2ecc71', '#e74c3c'], alpha=0.7)
plt.title("Độ đàn hồi của Insulin trước cơn sóng Glucose (Mảng phi tuyến)")
save_plot("Case4_Scatter_Insulin_Glucose")

###################################################################
# CASE 5: YẾU TỐ DI TRUYỀN (DPF)
###################################################################
plt.figure(figsize=(8,5))
sns.histplot(df_clinical['DiabetesPedigreeFunction'], kde=True, color='purple', bins=40)
plt.title("Phân phối Lệch phải (Skewed) đặc trưng của Di truyền học DPF")
save_plot("Case5_Hist_DPF")

plt.figure(figsize=(8,5))
sns.scatterplot(x='DiabetesPedigreeFunction', y='Glucose', hue='Outcome', data=df_clinical, palette=['#3498db', '#e74c3c'], alpha=0.7)
plt.axvline(1.0, color='black', linestyle='--', alpha=0.5)
plt.axhline(140, color='black', linestyle='--', alpha=0.5)
plt.title("Hệ nhị trục DPF vs Glucose (Vùng nguy hiểm: DPF > 1.0 & Glucose > 140)")
save_plot("Case5_Scatter_DPF_Glucose_RiskZones")

###################################################################
# CASE 6: THAI KỲ (PREGNANCIES)
###################################################################
plt.figure(figsize=(8,4))
sns.boxplot(x='Outcome', y='Pregnancies', data=df_clinical, palette=['#2ecc71', '#e74c3c'])
plt.title("Kiểm soát Outliers: Phân bổ số lần Mang thai theo sinh lý Outcome")
save_plot("Case6_Boxplot_Pregnancies")

###################################################################
# CASE 7: TƯƠNG QUAN DỮ LIỆU & ĐA CỘNG TUYẾN (VIF)
###################################################################
plt.figure(figsize=(10,8))
sns.heatmap(df_scaled[input_cols].corr(), annot=True, cmap='RdBu_r', fmt='.2f', square=True, vmin=-1, vmax=1)
plt.title("Ma trận tương quan Heatmap Pearson giữa 8 chỉ số lâm sàng")
save_plot("Case7_Heatmap_Correlation")

X_vif = df_scaled[input_cols]
vif_data = pd.DataFrame()
vif_data["Biến Input"] = X_vif.columns
corr_matrix = X_vif.corr().values
inv_corr = np.linalg.inv(corr_matrix)
vif_data["Chỉ số VIF"] = np.diag(inv_corr)

plt.figure(figsize=(9,5))
sns.barplot(x='Chỉ số VIF', y='Biến Input', data=vif_data, palette='rocket')
plt.axvline(2.0, color='red', linestyle='dashed', label='Ngưỡng an toàn tối thượng < 2')
plt.title("Kiểm định VIF - Giải quyết nghi ngờ Đa cộng tuyến")
plt.legend()
save_plot("Case7_VIF_Factor_Barplot")

###################################################################
# VÒNG LẶP MỞ RỘNG (SCATTER + KDE MARGINS) CHO 28 CẶP - ĐẠT >60 ẢNH TỔNG CỘNG
###################################################################
print("Đang kết xuất tổ hợp 28 đồ thị chéo để đẩy kho lưu trữ ảnh lên mức 60+ (Full Analysis)...")
pairs = list(itertools.combinations(input_cols, 2))
# Mỗi cặp sinh ra một Margin Joint Plot phức tạp trên nền Scaled Data
for x_col, y_col in pairs:
    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=x_col, y=y_col, hue=outcome_col, data=df_scaled, palette=['#3498db', '#e74c3c'], alpha=0.6, s=40)
    plt.title(f"Mạng tán xạ phân kỳ: {x_col} kết dính {y_col}")
    save_plot(f"Bonus_Combo_Scatter_{x_col}_vs_{y_col}")
    
for col in input_cols:
    plt.figure(figsize=(6, 5))
    sns.kdeplot(data=df_scaled, x=col, hue=outcome_col, fill=True, palette=['#3498db', '#e74c3c'])
    plt.title(f"Biến thiên phổ KDE độc lập phân nhóm: {col}")
    save_plot(f"Bonus_Single_KDE_{col}")

print("="*60)
print("✅ HOÀN THÀNH CHIẾN DỊCH KHÁM PHÁ DỮ LIỆU ĐAO TO BÚA LỚN!")
print(f"⭐ Đã Render thành công {img_counter - 1} biểu đồ Data Visualisation đỉnh cao.")
print(f"⭐ Mọi file đều được nén lại ở dạng (.png) sắc nét tại Directory: /{output_dir}")
print("="*60)